In [2]:
import pandas as pd
from pathlib import Path

In [3]:
BASE_DIR = Path.cwd().parent
events_path = BASE_DIR / "data/clean/events.parquet"

In [4]:
events_data = pd.read_parquet(events_path)

In [5]:
events_data.isna().sum()

event_datetime              0
event_name                  0
platform                    0
language                    0
user_id               5525689
user_pseudo_id              0
tour_id                633373
story_id              2749685
lang_id                801888
audio_time_played    13418334
audio_time_paused    13372689
dtype: int64

In [9]:
events_data.event_name.unique()

<ArrowStringArray>
[        'click_purchases_tab',                 'SCREEN_VIEW',
      'click_view_tickets_tab',            'click_listen_now',
                  'start_tour',                 'screen_view',
                 'story_start',             'collapse_player',
           'story_listened_20',           'story_listened_40',
           'story_listened_60',           'story_listened_80',
             'story_completed',                        'play',
                  'click_item',                 'click_story',
               'expand_player',                'click_3d_map',
              'click_location',                       'pause',
              'cmt_api_redeem',           'cmt_api_purchases',
               'cmt_api_tours',             'cmt_api_default',
               'cmt_api_error',           'tour_item_clicked',
          'deep_link_received',          'deep_link_response',
          'click_view_tickets',                  'next_story',
                 'backward_10',     

## These here are the event names. 

We can quickly deduce that some events are corellated with listening a tour while other events are corellated with navigating the app.

The most logical way of separating these would be that assuming that the data is correct: Listening events would be the ones without a tour id. The rest should be navigating events.

In [10]:
events_data.groupby('event_name').size().sort_values(ascending=False)

event_name
time_update               4784395
tour_download_progress    1216412
story_listened_20          976356
story_listened_40          902343
story_listened_60          851059
                           ...   
tap_explore                     5
tap_back                        2
add_to_cart                     2
purchase                        1
begin_checkout                  1
Length: 96, dtype: int64

We can see here that "time_update" is the most common event in the data. 

In [11]:
events_data.groupby('platform').size().sort_values(ascending=False)

platform
IOS        7914200
ANDROID    5557135
dtype: int64

IOS users are more than android users

Lets test to see whether the logic of navigating and listening events is correct

In [13]:
navigating_events = events_data.loc[events_data.tour_id.isna()]

In [14]:
navigating_events.event_name.unique()

<ArrowStringArray>
[        'click_purchases_tab',                 'SCREEN_VIEW',
      'click_view_tickets_tab',                 'screen_view',
              'cmt_api_redeem',           'cmt_api_purchases',
             'cmt_api_default',               'cmt_api_error',
          'deep_link_received',          'deep_link_response',
          'click_view_tickets',               'cmt_api_tours',
      'click_whatsapp_text_us',               'click_explore',
 'click_delete_account_button',               'click_call_us',
                  'click_cart',      'delete_account_attempt',
                   'view_item',            'click_search_bar',
         'click_main_response',                   'view_cart',
     'click_read_more_product',      'click_read_more_author',
       'click_product_section',                'swipe_images',
        'click_sign_up_button',             'sign_up_attempt',
     'click_skipsignin_button',              'signin_attempt',
         'click_signin_button',     

In [15]:
navigating_events.groupby('event_name').size().sort_values(ascending=False).to_csv("navigating_events.txt")

In [4]:
listening_events = events_data.loc[~events_data.tour_id.isna()]

In [5]:
listening_events.groupby('event_name').size().sort_values(ascending=False).to_csv("listening_events.txt")

## Lets analyze the events that appear in both

In [12]:

# paths
nav_path = "event_names/navigating_events.txt"
listen_path = "event_names/listening_events.txt"

# read txt files
nav = pd.read_csv(nav_path)
listen = pd.read_csv(listen_path)

events_both = (
    nav.merge(listen, on="event_name", how="inner", suffixes=("_navigating", "_listening"))
       .rename(columns={"0_navigating": "navigating_count",
                        "0_listening": "listening_count"})
)

events_both

,event_name,navigating_count,listening_count
0,screen_view,240090,232970
1,click_download_tour,6867,4027
2,tour_item_clicked,1072,150548
3,click_back,777,14882
4,cmt_api_tours,612,2235
5,collapse_player,112,172872
6,click_item,9,305649
7,tour_downloaded,5,10333
8,expand_player,4,49516
9,start_tour,3,64631


## Lets do the same but for story_id , maybe the problems overlap 

In [16]:
import gc

In [ ]:
# del story_events
# gc.collect()

## null story events

In [6]:
story_null_events = events_data.loc[events_data.story_id.isna()]

In [20]:
story_null_events.event_name.unique()

<ArrowStringArray>
[        'click_purchases_tab',                 'SCREEN_VIEW',
      'click_view_tickets_tab',            'click_listen_now',
                  'start_tour',                 'screen_view',
                  'click_item',                'click_3d_map',
              'click_location',              'cmt_api_redeem',
           'cmt_api_purchases',               'cmt_api_tours',
             'cmt_api_default',               'cmt_api_error',
           'tour_item_clicked',          'deep_link_received',
          'deep_link_response',          'click_view_tickets',
      'click_whatsapp_text_us',         'click_copy_ref_code',
               'click_explore', 'click_delete_account_button',
               'click_call_us',                  'click_back',
                  'click_cart',      'delete_account_attempt',
         'click_download_tour',       'tour_download_started',
      'tour_download_progress',         'tour_download_ended',
                   'view_item',     

In [21]:
story_null_events.groupby('event_name').size().sort_values(ascending=False).to_csv("story_null_events.txt")

## story events

In [22]:
story_events = events_data.loc[~events_data.story_id.isna()]

In [23]:
story_events.event_name.unique()

<ArrowStringArray>
[       'story_start',    'collapse_player',  'story_listened_20',
  'story_listened_40',  'story_listened_60',  'story_listened_80',
    'story_completed',               'play',        'click_story',
      'expand_player',              'pause',  'tour_item_clicked',
         'next_story',        'backward_10',         'forward_10',
        'time_update',         'click_item',     'previous_story',
 'click_progress_bar',   'click_multimedia',         'click_more']
Length: 21, dtype: str

In [24]:
story_events.groupby('event_name').size().sort_values(ascending=False).to_csv("story_events.txt")

## Lets analyze the events that appear in both

In [28]:

# paths
story_path = "event_names/story_events.txt"
story_null_path = "event_names/story_null_events.txt"

# read txt files
story = pd.read_csv(story_path)
story_null = pd.read_csv(story_null_path)

events_both = (
    story.merge(story_null, on="event_name", how="inner", suffixes=("_story", "_story_null"))
       .rename(columns={"0_story": "story_count",
                        "0_story_null": "story_null_count"})
)

events_both

,event_name,story_count,story_null_count
0,story_listened_20,976347,9
1,story_listened_40,902334,9
2,story_listened_60,851050,9
3,story_listened_80,808328,9
4,story_completed,620561,9
5,collapse_player,172872,112
6,tour_item_clicked,140835,10785
7,expand_player,49516,4
8,click_item,2221,303437


Lets check to see if the nulls are correlated

In [31]:
events_data.loc[
    events_data["story_id"].notna() & events_data["tour_id"].isna()
]

,event_datetime,event_name,platform,language,user_id,user_pseudo_id,tour_id,story_id,lang_id,audio_time_played,audio_time_paused


In [ ]:
story_null_events[story_null_events['event_name'] == "story_listened_20"]

,event_datetime,event_name,platform,language,user_id,user_pseudo_id,tour_id,story_id,lang_id,audio_time_played,audio_time_paused
538688,2025-07-05 07:04:05.259023,story_listened_20,ANDROID,en-us,<NA>,2b48dd062081a9bc8a0fe0957e99a1ea,278,<NA>,8,NaN,NaN
10648636,2025-10-22 17:37:58.333022,story_listened_20,ANDROID,el-gr,<NA>,daef455039f060ae1ef5f0a23d550d33,621,<NA>,2,NaN,NaN
11769158,2025-10-10 19:52:11.553104,story_listened_20,ANDROID,en-gb,<NA>,0420f9ede0c62572833d921f316a8e2d,535,<NA>,2,NaN,NaN
10276623,2025-10-24 06:50:47.119022,story_listened_20,ANDROID,el-gr,<NA>,6cdf33aae5fdc0695785a56c4199194c,893,<NA>,2,NaN,NaN
11153432,2025-10-24 06:51:06.913038,story_listened_20,ANDROID,el-gr,<NA>,6cdf33aae5fdc0695785a56c4199194c,908,<NA>,2,NaN,NaN
12930913,2025-10-24 08:45:05.502010,story_listened_20,ANDROID,el-gr,<NA>,6cdf33aae5fdc0695785a56c4199194c,908,<NA>,2,NaN,NaN
13172560,2025-10-24 10:39:37.181018,story_listened_20,ANDROID,el-gr,<NA>,6cdf33aae5fdc0695785a56c4199194c,908,<NA>,2,NaN,NaN
10912139,2025-10-24 11:37:57.252016,story_listened_20,ANDROID,el-gr,<NA>,6cdf33aae5fdc0695785a56c4199194c,908,<NA>,2,NaN,NaN
12833432,2025-10-24 12:26:18.289044,story_listened_20,ANDROID,el-gr,<NA>,6cdf33aae5fdc0695785a56c4199194c,908,<NA>,2,NaN,NaN


In [ ]:
events_data[events_data['user_pseudo_id'] == "2b48dd062081a9bc8a0fe0957e99a1ea"]

NameError: name 'events_data' is not defined